# Automobile Sales Analysis

## Phase 3 — Exploratory Data Analysis & Charts
    Libraries: Pandas, Matplotlib, Seaborn
    Input: automobile_sales_cleaned.csv
    Output: 8 charts saved to /visuals folder

Each chart answers one specific business question from the SQL analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# ── Chart style settings ──────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.dpi'      : 150,
    'figure.facecolor': 'white',
    'axes.facecolor'  : 'white',
    'font.family'     : 'sans-serif',
    'axes.titlesize'  : 14,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 10,
    'ytick.labelsize' : 10,
})

print("✓ Libraries loaded")
print("✓ visuals/ folder ready")

In [ ]:
df = pd.read_csv("C:/Users/anike/OneDrive/Desktop/Automobile sales power bi project/Datasheets/02_Automobile_Sales_cleaned.csv")
print("✓ File read successfully")
df['Date'] = pd.to_datetime(df['Date'])

print(f"✓ Data loaded | {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

In [ ]:
yearly = df.groupby('Year').agg(
    Revenue_bn  = ('Revenue', lambda x: round(x.sum() / 1e9, 2)),
    Total_Units = ('Units_Sold', 'sum')
).reset_index()

yearly['YoY_Growth'] = yearly['Revenue_bn'].pct_change() * 100

# ── CHART ─────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 5))

# Bar chart — revenue
colors = ['#2ecc71' if x >= 0 else '#e74c3c'
          for x in yearly['YoY_Growth'].fillna(0)]
bars = ax1.bar(yearly['Year'], yearly['Revenue_bn'],
               color='#2196F3', alpha=0.85, width=0.5, zorder=2)

# Line overlay — YoY growth % on secondary axis
ax2 = ax1.twinx()
ax2.plot(yearly['Year'], yearly['YoY_Growth'],
         color='#e74c3c', marker='o', linewidth=2,
         markersize=7, zorder=3, label='YoY Growth %')
ax2.axhline(0, color='#e74c3c', linewidth=0.8, linestyle='--', alpha=0.5)
ax2.set_ylabel('YoY Growth %', color='#e74c3c', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#e74c3c')

# Annotate revenue values on bars
for bar, val in zip(bars, yearly['Revenue_bn']):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f'₹{val}bn',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Annotate YoY % on line
for x, y in zip(yearly['Year'], yearly['YoY_Growth']):
    if pd.notna(y):
        ax2.annotate(f'{y:+.1f}%',
                     xy=(x, y),
                     xytext=(0, 9),
                     textcoords='offset points',
                     ha='center', fontsize=8.5,
                     color='#e74c3c', fontweight='bold')

# Labels & formatting
ax1.set_xlabel('Year', fontsize=11)
ax1.set_ylabel('Total Revenue (₹ Billion)', fontsize=11)
ax1.set_title('Annual Revenue Trend with YoY Growth (2020–2025)',
              fontsize=14, fontweight='bold', pad=15)
ax1.set_xticks(yearly['Year'])
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0fbn'))
ax1.set_ylim(0, yearly['Revenue_bn'].max() * 1.2)
ax1.grid(axis='y', alpha=0.4, zorder=0)

plt.tight_layout()
plt.savefig('01_annual_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/01_annual_revenue_trend.png")

In [ ]:
cat_yearly = df.groupby(['Year', 'Category'])['Revenue'].sum().div(1e9).round(2)
cat_yearly = cat_yearly.unstack('Category').fillna(0)

# Consistent category colors throughout all charts
CAT_COLORS = {
    'SUV'      : '#2196F3',
    'Sedan'    : '#FF9800',
    'Electric' : '#4CAF50',
    'Hatchback': '#9C27B0'
}

# ── CHART ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

cat_yearly.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=[CAT_COLORS[c] for c in cat_yearly.columns],
    width=0.55,
    alpha=0.9,
    zorder=2
)

# Annotate total revenue on top of each stacked bar
totals = cat_yearly.sum(axis=1)
for i, (year, total) in enumerate(totals.items()):
    ax.text(i, total + 0.3, f'₹{total:.1f}bn',
            ha='center', va='bottom',
            fontsize=8.5, fontweight='bold', color='#333333')

# Labels & formatting
ax.set_title('Annual Revenue by Vehicle Category (2020–2025)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Revenue (₹ Billion)', fontsize=11)
ax.set_xticklabels(cat_yearly.index, rotation=0)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0fbn'))
ax.legend(title='Category', bbox_to_anchor=(1.01, 1),
          loc='upper left', frameon=True)
ax.grid(axis='y', alpha=0.4, zorder=0)
ax.set_ylim(0, totals.max() * 1.12)

plt.tight_layout()
plt.savefig('02_revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/02_revenue_by_category.png")

In [ ]:
fuel_yearly = (
    df.groupby(['Year', 'Fuel_Type'])['Units_Sold']
    .sum()
    .unstack('Fuel_Type')
    .fillna(0)
)

# Convert to percentage share
fuel_share = fuel_yearly.div(fuel_yearly.sum(axis=1), axis=0) * 100

FUEL_COLORS = {
    'Petrol'  : '#FF9800',
    'Diesel'  : '#607D8B',
    'Electric': '#4CAF50'
}

# ── CHART ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

for fuel in ['Petrol', 'Diesel', 'Electric']:
    ax.plot(fuel_share.index, fuel_share[fuel],
            marker='o', linewidth=2.5, markersize=8,
            color=FUEL_COLORS[fuel], label=fuel, zorder=3)

    # Annotate each data point
    for year, val in fuel_share[fuel].items():
        offset = 8 if fuel == 'Petrol' else -14
        ax.annotate(f'{val:.1f}%',
                    xy=(year, val),
                    xytext=(0, offset),
                    textcoords='offset points',
                    ha='center', fontsize=8,
                    color=FUEL_COLORS[fuel], fontweight='bold')

# Highlight the flat EV band
ax.axhspan(8, 11, alpha=0.08, color='#4CAF50', zorder=0)
ax.text(2024.6, 9.5, 'EV flat zone\n(9–10%)',
        fontsize=8, color='#4CAF50', alpha=0.8,
        ha='center', style='italic')

# Labels & formatting
ax.set_title('Fuel Type Unit Share Trend (2020–2025)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Share of Total Units Sold (%)', fontsize=11)
ax.set_xticks(fuel_share.index)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_ylim(0, 60)
ax.legend(title='Fuel Type', frameon=True, fontsize=10)
ax.grid(axis='y', alpha=0.4, zorder=0)

plt.tight_layout()
plt.savefig('03_fuel_type_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/03_fuel_type_trend.png")

In [ ]:
dealer_summary = df.groupby('Dealer').agg(
    Revenue_bn      = ('Revenue', lambda x: round(x.sum() / 1e9, 2)),
    Avg_Price_Lakhs = ('Unit_Price', lambda x: round(x.mean() / 1e5, 2)),
    Avg_Satisfaction= ('Customer_Satisfaction_Score', 'mean')
).reset_index().sort_values('Revenue_bn', ascending=True)

# Tier classification
def get_tier(price):
    if price >= 25:   return 'Premium (₹25L+)'
    elif price >= 18: return 'Mid-Range (₹18–25L)'
    else:             return 'Budget (Below ₹18L)'

dealer_summary['Tier'] = dealer_summary['Avg_Price_Lakhs'].apply(get_tier)

TIER_COLORS = {
    'Premium (₹25L+)'     : '#4CAF50',
    'Mid-Range (₹18–25L)' : '#2196F3',
    'Budget (Below ₹18L)' : '#FF9800'
}

# ── CHART ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(
    dealer_summary['Dealer'],
    dealer_summary['Revenue_bn'],
    color=[TIER_COLORS[t] for t in dealer_summary['Tier']],
    alpha=0.88, height=0.6, zorder=2
)

# Annotate revenue + avg price on each bar
for bar, (_, row) in zip(bars, dealer_summary.iterrows()):
    ax.text(bar.get_width() + 0.2,
            bar.get_y() + bar.get_height() / 2,
            f'₹{row.Revenue_bn}bn  |  avg ₹{row.Avg_Price_Lakhs}L',
            va='center', fontsize=8.5, color='#333333')

# Legend for tiers
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=TIER_COLORS[t], label=t)
    for t in TIER_COLORS
]
ax.legend(handles=legend_elements, title='Price Tier',
          loc='lower right', frameon=True, fontsize=9)

# Labels & formatting
ax.set_title('Dealer Revenue Ranking by Price Tier',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Revenue (₹ Billion)', fontsize=11)
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0fbn'))
ax.set_xlim(0, dealer_summary['Revenue_bn'].max() * 1.45)
ax.grid(axis='x', alpha=0.4, zorder=0)

plt.tight_layout()
plt.savefig('04_dealer_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/04_dealer_ranking.png")

In [ ]:
model_rev = (
    df.groupby(['Vehicle_Model', 'Category'])['Revenue']
    .sum()
    .div(1e9)
    .round(2)
    .reset_index()
    .sort_values('Revenue', ascending=False)
    .reset_index(drop=True)
)

model_rev['Cumulative_Pct'] = (
    model_rev['Revenue'].cumsum() / model_rev['Revenue'].sum() * 100
)

# Color bars by category using same palette as Chart 2
bar_colors = [CAT_COLORS[c] for c in model_rev['Category']]

# ── CHART ─────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(16, 6))

# Bars — revenue per model
bars = ax1.bar(
    range(len(model_rev)),
    model_rev['Revenue'],
    color=bar_colors, alpha=0.85,
    width=0.7, zorder=2
)

# Line — cumulative %
ax2 = ax1.twinx()
ax2.plot(
    range(len(model_rev)),
    model_rev['Cumulative_Pct'],
    color='#e74c3c', linewidth=2,
    marker='o', markersize=3, zorder=3
)

# 80% reference line
ax2.axhline(80, color='#e74c3c', linestyle='--',
            linewidth=1.2, alpha=0.7, zorder=1)
ax2.text(len(model_rev) - 1, 81.5, '80% Revenue',
         color='#e74c3c', fontsize=9,
         ha='right', fontweight='bold')

# Mark the 80% cutoff bar
cutoff_idx = model_rev[model_rev['Cumulative_Pct'] <= 80].index[-1]
ax1.axvline(cutoff_idx, color='#333333', linestyle='--',
            linewidth=1.2, alpha=0.6, zorder=1)
ax1.text(cutoff_idx + 0.3, model_rev['Revenue'].max() * 0.85,
         f'Model #{cutoff_idx + 1}\n({cutoff_idx + 1} of 47 models\n= 80% revenue)',
         fontsize=8, color='#333333',
         bbox=dict(boxstyle='round,pad=0.3',
                   facecolor='white', edgecolor='#cccccc'))

# X axis — model names
ax1.set_xticks(range(len(model_rev)))
ax1.set_xticklabels(model_rev['Vehicle_Model'],
                    rotation=90, fontsize=7.5)

# Category legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=CAT_COLORS[c], label=c)
                   for c in CAT_COLORS]
ax1.legend(handles=legend_elements, title='Category',
           loc='upper left', frameon=True, fontsize=9)

# Labels & formatting
ax1.set_title('Vehicle Model Pareto Analysis — Revenue Contribution',
              fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Vehicle Model (ranked by revenue)', fontsize=11)
ax1.set_ylabel('Revenue (₹ Billion)', fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0fbn'))
ax2.set_ylabel('Cumulative Revenue %', color='#e74c3c', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#e74c3c')
ax2.set_ylim(0, 115)
ax1.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
##plt.savefig('05_pareto_models.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/05_pareto_models.png")

In [ ]:
region_heat = (
    df.groupby(['Year', 'Region'])['Revenue']
    .sum()
    .div(1e9)
    .round(2)
    .unstack('Region')
)

# ── CHART ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

sns.heatmap(
    region_heat,
    ax=ax,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Revenue (₹ Billion)', 'shrink': 0.8},
    annot_kws={'size': 10, 'weight': 'bold'}
)

# Labels & formatting
ax.set_title('Regional Revenue Heatmap (₹ Billion) — 2020 to 2025',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Region', fontsize=11)
ax.set_ylabel('Year', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('06_regional_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/06_regional_heatmap.png")

In [ ]:
dealer_scatter = df.groupby('Dealer').agg(
    Revenue_bn      = ('Revenue', lambda x: round(x.sum() / 1e9, 2)),
    Avg_Satisfaction= ('Customer_Satisfaction_Score', 'mean'),
    Avg_Price_Lakhs = ('Unit_Price', lambda x: round(x.mean() / 1e5, 2))
).reset_index()

# Quadrant thresholds — dataset averages
sat_mid = dealer_scatter['Avg_Satisfaction'].mean()
rev_mid = dealer_scatter['Revenue_bn'].mean()

# Tier for colour
def get_tier(price):
    if price >= 25:   return 'Premium (₹25L+)'
    elif price >= 18: return 'Mid-Range (₹18–25L)'
    else:             return 'Budget (Below ₹18L)'

dealer_scatter['Tier'] = dealer_scatter['Avg_Price_Lakhs'].apply(get_tier)

TIER_COLORS = {
    'Premium (₹25L+)'     : '#4CAF50',
    'Mid-Range (₹18–25L)' : '#2196F3',
    'Budget (Below ₹18L)' : '#FF9800'
}

# ── CHART ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for tier, group in dealer_scatter.groupby('Tier'):
    ax.scatter(
        group['Avg_Satisfaction'],
        group['Revenue_bn'],
        c=TIER_COLORS[tier],
        s=120, alpha=0.85,
        label=tier, zorder=3,
        edgecolors='white', linewidth=0.8
    )

# Annotate dealer names
for _, row in dealer_scatter.iterrows():
    ax.annotate(
        row['Dealer'],
        xy=(row['Avg_Satisfaction'], row['Revenue_bn']),
        xytext=(5, 4),
        textcoords='offset points',
        fontsize=7.5, color='#333333'
    )

# Quadrant lines
ax.axvline(sat_mid, color='#999999', linestyle='--',
           linewidth=1, alpha=0.7, zorder=1)
ax.axhline(rev_mid, color='#999999', linestyle='--',
           linewidth=1, alpha=0.7, zorder=1)

# Quadrant labels
quad_style = dict(fontsize=8.5, alpha=0.5, fontweight='bold')
ax.text(sat_mid + 0.01, dealer_scatter['Revenue_bn'].max() * 0.97,
        'High Satisfaction\nHigh Revenue', color='#2ecc71', **quad_style)
ax.text(dealer_scatter['Avg_Satisfaction'].min(),
        dealer_scatter['Revenue_bn'].max() * 0.97,
        'Low Satisfaction\nHigh Revenue', color='#e74c3c', **quad_style)
ax.text(sat_mid + 0.01, rev_mid * 0.25,
        'High Satisfaction\nLow Revenue', color='#3498db', **quad_style)
ax.text(dealer_scatter['Avg_Satisfaction'].min(), rev_mid * 0.25,
        'Low Satisfaction\nLow Revenue', color='#999999', **quad_style)

# Labels & formatting
ax.set_title('Customer Satisfaction vs Revenue — Dealer Quadrant Analysis',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Average Customer Satisfaction Score', fontsize=11)
ax.set_ylabel('Total Revenue (₹ Billion)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0fbn'))
ax.legend(title='Price Tier', frameon=True, fontsize=9)
ax.grid(alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('07_satisfaction_vs_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/07_satisfaction_vs_revenue.png")

In [ ]:
month_heat = (
    df.groupby(['Year', 'Month'])['Revenue']
    .sum()
    .div(1e9)
    .round(2)
    .unstack('Month')
)

# Rename columns from numbers to month abbreviations
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
month_heat.rename(columns=month_names, inplace=True)

# ── CHART ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

sns.heatmap(
    month_heat,
    ax=ax,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Revenue (₹ Billion)', 'shrink': 0.8},
    annot_kws={'size': 9, 'weight': 'bold'}
)

# Highlight peak cell per year with a border
for year_idx, year in enumerate(month_heat.index):
    peak_month = month_heat.loc[year].idxmax()
    peak_col_idx = list(month_heat.columns).index(peak_month)
    ax.add_patch(plt.Rectangle(
        (peak_col_idx, year_idx), 1, 1,
        fill=False, edgecolor='#2196F3',
        lw=2.5, zorder=4
    ))

# Labels & formatting
ax.set_title('Monthly Revenue Seasonality Heatmap (₹ Billion)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Year', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Add note about blue border
ax.text(0, -0.6, '🔵 Blue border = peak month for that year',
        transform=ax.transAxes, fontsize=8.5,
        color='#2196F3', style='italic')

plt.tight_layout()
plt.savefig('08_monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → visuals/08_monthly_seasonality.png")